# Cart2Insights - Business Insights & Recommendations

This notebook presents the key business insights obtained from the
Cart2Insights e-commerce analysis.

The insights are derived from:

- Exploratory Data Analysis
- Feature Engineering
- Statistical Analysis
- SQL Business Analysis
- Customer Analysis
- Seller Analysis
- Product and Category Analysis
- Delivery Analysis
- Customer Experience Analysis

The objective is to convert analytical findings into actionable
business recommendations.

Each major finding is interpreted using the following structure:

**Observation → Interpretation → Business Impact → Recommendation**

## Business Context

Cart2Insights analyzes an e-commerce platform across multiple business
dimensions including sales, customers, products, sellers, payments,
delivery performance, and customer reviews.

The key business questions addressed in this analysis are:

1. How is the overall business performing?
2. How does revenue change over time?
3. Which product categories generate the most revenue?
4. Which products perform best?
5. How do customers behave and spend?
6. How important are repeat customers?
7. Which sellers contribute the most revenue?
8. How efficient is the delivery process?
9. Does delivery performance relate to customer satisfaction?
10. Which product categories receive higher customer ratings?
11. What statistical evidence supports the business findings?
12. What actions can the business take based on these findings?

Code: Database Connection

In [1]:
import os
import pandas as pd
import numpy as np

from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from urllib.parse import quote_plus

load_dotenv("../.env")

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME")

password = quote_plus(DB_PASSWORD)

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{password}@"
    f"{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Database connection created successfully.")

c:\Users\Zoya khan\AppData\Local\Programs\Python\Python37\lib\site-packages\pymysql\_auth.py:9: CryptographyDeprecationWarning: Python 3.7 is no longer supported by the Python core team and support for it is deprecated in cryptography. The next release of cryptography will remove support for Python 3.7.
  from cryptography.hazmat.backends import default_backend


Database connection created successfully.


Code: Connection Test

In [2]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))

print("MySQL connection successful.")

MySQL connection successful.


INSIGHT 1 — OVERALL BUSINESS PERFORMANCE

 1. Overall Business Performance

The first analysis evaluates the overall scale and performance of the
e-commerce business using revenue, order volume, customer count,
seller count, average order value, and customer review score.

In [3]:
overall_query = """
SELECT
    ROUND(SUM(order_total_value), 2) AS total_revenue,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT customer_unique_id) AS total_customers
FROM order_features;
"""

overall_df = pd.read_sql(overall_query, engine)

seller_query = """
SELECT COUNT(DISTINCT seller_id) AS total_sellers
FROM sellers;
"""

seller_df = pd.read_sql(seller_query, engine)

review_query = """
SELECT
    ROUND(AVG(review_score), 2) AS average_review_score,
    COUNT(*) AS total_reviews
FROM order_reviews
WHERE review_score IS NOT NULL;
"""

review_df = pd.read_sql(review_query, engine)

total_revenue = overall_df["total_revenue"].iloc[0]
total_orders = overall_df["total_orders"].iloc[0]
total_customers = overall_df["total_customers"].iloc[0]
total_sellers = seller_df["total_sellers"].iloc[0]
average_review_score = review_df["average_review_score"].iloc[0]
total_reviews = review_df["total_reviews"].iloc[0]

average_order_value = total_revenue / total_orders

overall_summary = pd.DataFrame({
    "Metric": [
        "Total Revenue",
        "Total Orders",
        "Total Customers",
        "Total Sellers",
        "Average Order Value",
        "Average Review Score",
        "Total Reviews"
    ],
    "Value": [
        total_revenue,
        total_orders,
        total_customers,
        total_sellers,
        round(average_order_value, 2),
        average_review_score,
        total_reviews
    ]
})

overall_summary

,Metric,Value
0,Total Revenue,15843553.24
1,Total Orders,99441.00
2,Total Customers,96096.00
3,Total Sellers,3095.00
4,Average Order Value,159.33
5,Average Review Score,4.09
6,Total Reviews,99224.00


In [4]:
# generate insight
print("OBSERVATION")
print(
    f"The platform generated total revenue of {total_revenue:,.2f} "
    f"across {total_orders:,} orders and served {total_customers:,} "
    f"unique customers."
)

print("\nINTERPRETATION")
print(
    f"The average order value was {average_order_value:,.2f}, while "
    f"the average customer review score was {average_review_score:.2f} "
    f"out of 5."
)

print("\nBUSINESS IMPACT")
print(
    "These KPIs provide a baseline for evaluating overall business "
    "performance and monitoring future changes in sales, customer "
    "activity, and customer satisfaction."
)

print("\nRECOMMENDATION")
print(
    "The business should monitor revenue, order volume, average order "
    "value, customer growth, and customer satisfaction together rather "
    "than relying on revenue alone."
)

OBSERVATION
The platform generated total revenue of 15,843,553.24 across 99,441 orders and served 96,096 unique customers.

INTERPRETATION
The average order value was 159.33, while the average customer review score was 4.09 out of 5.

BUSINESS IMPACT
These KPIs provide a baseline for evaluating overall business performance and monitoring future changes in sales, customer activity, and customer satisfaction.

RECOMMENDATION
The business should monitor revenue, order volume, average order value, customer growth, and customer satisfaction together rather than relying on revenue alone.


INSIGHT 2 — MONTHLY REVENUE TREND

 2. Monthly Revenue Trend

Monthly revenue analysis helps identify periods of stronger and weaker
sales performance.

The analysis can support decisions related to marketing campaigns,
inventory planning, seller capacity, and operational resources.

In [5]:
monthly_query = """
WITH monthly_sales AS (
    SELECT
        DATE_FORMAT(o.order_purchase_timestamp, '%%Y-%%m') AS month,
        ROUND(
            SUM(oi.price + oi.freight_value),
            2
        ) AS revenue
    FROM orders o
    JOIN order_items oi
        ON o.order_id = oi.order_id
    GROUP BY month
)

SELECT
    month,
    revenue
FROM monthly_sales
ORDER BY month;
"""

monthly_df = pd.read_sql(monthly_query, engine)

monthly_df

,month,revenue
0,2016-09,354.75
1,2016-10,56808.84
2,2016-12,19.62
3,2017-01,137188.49
4,2017-02,286280.62
5,2017-03,432048.59
6,2017-04,412422.24
7,2017-05,586190.95
8,2017-06,502963.04
9,2017-07,584971.62


In [6]:
#highest and lowest month
highest_month = monthly_df.loc[
    monthly_df["revenue"].idxmax()
]

lowest_month = monthly_df.loc[
    monthly_df["revenue"].idxmin()
]

print("HIGHEST REVENUE MONTH")
print(
    f"{highest_month['month']} generated "
    f"{highest_month['revenue']:,.2f} in revenue."
)

print("\nLOWEST REVENUE MONTH")
print(
    f"{lowest_month['month']} generated "
    f"{lowest_month['revenue']:,.2f} in revenue."
)

print("\nOBSERVATION")
print(
    f"The highest monthly revenue occurred in {highest_month['month']}, "
    f"while the lowest monthly revenue occurred in "
    f"{lowest_month['month']}."
)

print("\nINTERPRETATION")
print(
    "Revenue varies across time, indicating differences in customer "
    "purchasing activity and demand."
)

print("\nBUSINESS IMPACT")
print(
    "Understanding high- and low-performing periods can help the "
    "business plan marketing campaigns, inventory, seller capacity, "
    "and logistics resources."
)

print("\nRECOMMENDATION")
print(
    "The business should investigate the reasons behind high-revenue "
    "periods and use these patterns to improve future campaign and "
    "inventory planning."
)

HIGHEST REVENUE MONTH
2017-11 generated 1,179,143.77 in revenue.

LOWEST REVENUE MONTH
2016-12 generated 19.62 in revenue.

OBSERVATION
The highest monthly revenue occurred in 2017-11, while the lowest monthly revenue occurred in 2016-12.

INTERPRETATION
Revenue varies across time, indicating differences in customer purchasing activity and demand.

BUSINESS IMPACT
Understanding high- and low-performing periods can help the business plan marketing campaigns, inventory, seller capacity, and logistics resources.

RECOMMENDATION
The business should investigate the reasons behind high-revenue periods and use these patterns to improve future campaign and inventory planning.


INSIGHT 3 — PRODUCT CATEGORY PERFORMANCE

 3. Product Category Performance

Product category analysis identifies the categories that contribute
most strongly to revenue, order volume, and product demand.

Understanding category-level performance can support inventory,
marketing, and seller strategy.

In [7]:
category_query = """
SELECT
    category_name,
    product_count,
    items_sold,
    order_count,
    category_revenue
FROM category_features
ORDER BY category_revenue DESC;
"""

category_df = pd.read_sql(category_query, engine)

category_df.head(15)

,category_name,product_count,items_sold,order_count,category_revenue
0,health_beauty,2444,9670,8836,1258681.34
1,watches_gifts,1329,5991,5624,1205005.68
2,bed_bath_table,3029,11115,9417,1036988.68
3,sports_leisure,2867,8641,7720,988048.97
4,computers_accessories,1639,7827,6689,911954.32
5,furniture_decor,2657,8334,6449,729762.49
6,cool_stuff,789,3796,3632,635290.85
7,housewares,2335,6964,5884,632248.66
8,auto,1900,4235,3897,592720.11
9,garden_tools,753,4347,3518,485256.46


In [8]:
top_category = category_df.iloc[0]

print("TOP PERFORMING CATEGORY")
print(
    f"{top_category['category_name']} generated "
    f"{top_category['category_revenue']:,.2f} in revenue."
)

print("\nOBSERVATION")
print(
    f"The highest-revenue product category is "
    f"{top_category['category_name']}."
)

print("\nINTERPRETATION")
print(
    "Product demand and revenue are not distributed equally across "
    "all product categories."
)

print("\nBUSINESS IMPACT")
print(
    "High-performing categories have a greater influence on overall "
    "business revenue and therefore require appropriate inventory "
    "and seller support."
)

print("\nRECOMMENDATION")
print(
    "The business should prioritize high-performing categories for "
    "inventory planning, promotions, and seller acquisition while "
    "monitoring lower-performing categories for improvement opportunities."
)

TOP PERFORMING CATEGORY
health_beauty generated 1,258,681.34 in revenue.

OBSERVATION
The highest-revenue product category is health_beauty.

INTERPRETATION
Product demand and revenue are not distributed equally across all product categories.

BUSINESS IMPACT
High-performing categories have a greater influence on overall business revenue and therefore require appropriate inventory and seller support.

RECOMMENDATION
The business should prioritize high-performing categories for inventory planning, promotions, and seller acquisition while monitoring lower-performing categories for improvement opportunities.


INSIGHT 4 — TOP PRODUCTS

4. Top-Performing Products

Product-level analysis identifies the products that generate the
highest revenue and sales volume.

These products can be important drivers of overall business performance.

In [9]:
top_products_query = """
SELECT
    product_id,
    category_english,
    product_revenue,
    items_sold,
    order_count
FROM product_features
ORDER BY product_revenue DESC
LIMIT 10;
"""

top_products_df = pd.read_sql(
    top_products_query,
    engine
)

top_products_df

,product_id,category_english,product_revenue,items_sold,order_count
0,bb50f2e236e5eea0100680137654686c,health_beauty,63885.00,195,187
1,6cdd53843498f92890544667809f1595,health_beauty,54730.20,156,151
2,d6160fb7873f184099d9bc95e30376af,computers,48899.34,35,35
3,d1c427060a0f73f6b889a5c7c61f2ac4,computers_accessories,47214.51,343,323
4,99a4788cb24856965c36a24e339b6058,bed_bath_table,43025.56,488,467
5,3dd2a17168ec895c781a9191c1e95ad7,computers_accessories,41082.60,274,255
6,25c38557cf793876c5abdd5931f922db,baby,38907.32,38,38
7,5f504b3a1c75b73d6151be81eb05bdc9,cool_stuff,37733.90,63,63
8,53b36df67ebb7c41585e8d54d6772e08,watches_gifts,37683.42,323,306
9,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,37608.90,527,431


In [10]:
top_product = top_products_df.iloc[0]

print("TOP PRODUCT")
print(
    f"Product {top_product['product_id']} generated "
    f"{top_product['product_revenue']:,.2f} in revenue."
)

print("\nOBSERVATION")
print(
    "The top-performing products contribute a significant amount "
    "of revenue compared with other individual products."
)

print("\nINTERPRETATION")
print(
    "Customer demand is concentrated among certain products, making "
    "their availability important for maintaining sales performance."
)

print("\nBUSINESS IMPACT")
print(
    "Stock-outs or operational issues affecting high-performing "
    "products could directly affect revenue."
)

print("\nRECOMMENDATION")
print(
    "The business should prioritize inventory availability and "
    "seller performance monitoring for high-revenue products."
)

TOP PRODUCT
Product bb50f2e236e5eea0100680137654686c generated 63,885.00 in revenue.

OBSERVATION
The top-performing products contribute a significant amount of revenue compared with other individual products.

INTERPRETATION
Customer demand is concentrated among certain products, making their availability important for maintaining sales performance.

BUSINESS IMPACT
Stock-outs or operational issues affecting high-performing products could directly affect revenue.

RECOMMENDATION
The business should prioritize inventory availability and seller performance monitoring for high-revenue products.


INSIGHT 5 — CUSTOMER SPENDING

 5. Customer Spending and Customer Value

Customer-level analysis helps identify high-value customers based on
total spending, order frequency, and average order value.

This analysis can support customer retention and personalization
strategies.

In [11]:
customer_query = """
SELECT
    customer_unique_id,
    customer_order_count,
    customer_total_spending,
    average_order_value,
    repeat_customer_indicator
FROM customer_features
ORDER BY customer_total_spending DESC;
"""

customer_df = pd.read_sql(
    customer_query,
    engine
)

customer_df.head(10)

,customer_unique_id,customer_order_count,customer_total_spending,average_order_value,repeat_customer_indicator
0,0a0a92112bd4c708ca5fde585afaa872,1,13664.08,13664.08,0
1,da122df9eeddfedc1dc1f5349a1a690c,2,7571.63,3785.82,1
2,763c8b1c9c68a0229c42c9fc6f662b93,1,7274.88,7274.88,0
3,dc4802a71eae9be1dd28f5d788ceb526,1,6929.31,6929.31,0
4,459bef486812aa25204be022145caa62,1,6922.21,6922.21,0
5,ff4159b92c40ebe40454e3e6a7c35ed6,1,6726.66,6726.66,0
6,4007669dec559734d6f53e029e360987,1,6081.54,6081.54,0
7,5d0a2980b292d049061542014e8960bf,1,4809.44,4809.44,0
8,eebb5dda148d3893cdaf5b5ca3040ccb,1,4764.34,4764.34,0
9,48e1ac109decbb87765a3eade6854098,1,4681.78,4681.78,0


In [12]:
top_customer = customer_df.iloc[0]

print("TOP CUSTOMER BY SPENDING")
print(
    f"Customer {top_customer['customer_unique_id']} has total "
    f"spending of {top_customer['customer_total_spending']:,.2f}."
)

print("\nOBSERVATION")
print(
    "Customer spending varies considerably across the customer base."
)

print("\nINTERPRETATION")
print(
    "A group of customers contributes higher spending and therefore "
    "represents a higher-value segment."
)

print("\nBUSINESS IMPACT")
print(
    "High-value customers are important for revenue retention and "
    "long-term customer value."
)

print("\nRECOMMENDATION")
print(
    "The business should identify high-value customers and use "
    "targeted retention strategies, personalized offers, and "
    "relevant product recommendations."
)

TOP CUSTOMER BY SPENDING
Customer 0a0a92112bd4c708ca5fde585afaa872 has total spending of 13,664.08.

OBSERVATION
Customer spending varies considerably across the customer base.

INTERPRETATION
A group of customers contributes higher spending and therefore represents a higher-value segment.

BUSINESS IMPACT
High-value customers are important for revenue retention and long-term customer value.

RECOMMENDATION
The business should identify high-value customers and use targeted retention strategies, personalized offers, and relevant product recommendations.


INSIGHT 6 — REPEAT CUSTOMERS

6. Repeat Customer Behavior

Repeat customer behavior is an important indicator of customer
retention.

The repeat customer indicator created during feature engineering is
used to compare repeat customers with one-time customers.

In [13]:
repeat_query = """
SELECT
    CASE
        WHEN repeat_customer_indicator = 1
            THEN 'Repeat Customer'
        ELSE 'One-Time Customer'
    END AS customer_type,
    COUNT(*) AS customer_count
FROM customer_features
GROUP BY repeat_customer_indicator;
"""

repeat_df = pd.read_sql(
    repeat_query,
    engine
)

repeat_df

,customer_type,customer_count
0,One-Time Customer,93099
1,Repeat Customer,2997


In [14]:
total_customer_count = repeat_df["customer_count"].sum()

repeat_count = repeat_df.loc[
    repeat_df["customer_type"] == "Repeat Customer",
    "customer_count"
].sum()

one_time_count = repeat_df.loc[
    repeat_df["customer_type"] == "One-Time Customer",
    "customer_count"
].sum()

repeat_percentage = (
    repeat_count / total_customer_count
) * 100

one_time_percentage = (
    one_time_count / total_customer_count
) * 100

print("CUSTOMER RETENTION ANALYSIS")
print(
    f"Repeat customers: {repeat_count:,} "
    f"({repeat_percentage:.2f}%)"
)

print(
    f"One-time customers: {one_time_count:,} "
    f"({one_time_percentage:.2f}%)"
)

print("\nOBSERVATION")
print(
    f"Repeat customers represent {repeat_percentage:.2f}% "
    f"of the analyzed customer base."
)

print("\nINTERPRETATION")
print(
    "The repeat-customer share provides an indication of customer "
    "retention and purchasing loyalty."
)

print("\nBUSINESS IMPACT")
print(
    "Increasing repeat purchases can generate additional revenue "
    "from existing customers and strengthen long-term customer value."
)

print("\nRECOMMENDATION")
print(
    "The business should use targeted retention campaigns, "
    "personalized recommendations, loyalty initiatives, and "
    "post-purchase engagement to encourage repeat purchases."
)

CUSTOMER RETENTION ANALYSIS
Repeat customers: 2,997 (3.12%)
One-time customers: 93,099 (96.88%)

OBSERVATION
Repeat customers represent 3.12% of the analyzed customer base.

INTERPRETATION
The repeat-customer share provides an indication of customer retention and purchasing loyalty.

BUSINESS IMPACT
Increasing repeat purchases can generate additional revenue from existing customers and strengthen long-term customer value.

RECOMMENDATION
The business should use targeted retention campaigns, personalized recommendations, loyalty initiatives, and post-purchase engagement to encourage repeat purchases.


INSIGHT 7 — SELLER PERFORMANCE

7. Seller Performance

Seller analysis evaluates revenue, order volume, and items sold to
identify high-performing sellers.

Seller performance is important because sellers directly influence
product availability, sales volume, and customer experience.

In [15]:
seller_query = """
SELECT
    seller_id,
    seller_revenue,
    seller_order_count,
    items_sold,
    average_item_price
FROM seller_features
ORDER BY seller_revenue DESC;
"""

seller_df = pd.read_sql(
    seller_query,
    engine
)

seller_df.head(10)

,seller_id,seller_revenue,seller_order_count,items_sold,average_item_price
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1132,1156,198.51
1,53243585a1d6dc2643021fd1853d8905,222776.05,358,410,543.36
2,4a3ca9315b744ce9f8e9374361493884,200472.92,1806,1987,100.89
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,585,586,331.13
4,7c67e1448b00f6e969d365cea6b010ab,187923.89,982,1364,137.77
5,7e93a43ef30c4f03f38b393420bc753a,176431.87,336,340,518.92
6,da8622b14eb17ae2831f4ac5b9dab84a,160236.57,1314,1551,103.31
7,7a67c85e85bb2ce8582c35f2203ad736,141745.53,1160,1171,121.05
8,1025f0e2d44d7041d6cf58b6550e0bfa,138968.55,915,1428,97.32
9,955fee9216a65b617aa5c0531780ce60,135171.70,1287,1499,90.17


In [18]:
top_seller = seller_df.iloc[0]

print("TOP SELLER")
print(
    f"Seller {top_seller['seller_id']} generated "
    f"{top_seller['seller_revenue']:,.2f} in revenue."
)

print("\nOBSERVATION")
print(
    "Seller performance varies substantially based on revenue, "
    "order volume, and items sold."
)

print("\nINTERPRETATION")
print(
    "Some sellers contribute considerably more to overall marketplace "
    "sales than others."
)

print("\nBUSINESS IMPACT")
print(
    "High-performing sellers can have a significant effect on "
    "overall platform revenue."
)

print("\nRECOMMENDATION")
print(
    "The business should monitor seller revenue, order volume, "
    "items sold, and customer ratings together. High-performing "
    "sellers can be encouraged to expand their product selection "
    "while underperforming sellers can be evaluated for improvement."
)

TOP SELLER
Seller 4869f7a5dfa277a7dca6462dcf3b52b2 generated 229,472.63 in revenue.

OBSERVATION
Seller performance varies substantially based on revenue, order volume, and items sold.

INTERPRETATION
Some sellers contribute considerably more to overall marketplace sales than others.

BUSINESS IMPACT
High-performing sellers can have a significant effect on overall platform revenue.

RECOMMENDATION
The business should monitor seller revenue, order volume, items sold, and customer ratings together. High-performing sellers can be encouraged to expand their product selection while underperforming sellers can be evaluated for improvement.


INSIGHT 8 — DELIVERY PERFORMANCE

8. Delivery Performance

Delivery performance is analyzed using delivery days, delivery delay,
and delivery status.

The analysis identifies the proportion of orders delivered early,
on time, late, or not delivered.

In [19]:
delivery_query = """
SELECT
    delivery_status,
    COUNT(*) AS order_count,
    ROUND(AVG(delivery_days), 2) AS average_delivery_days,
    ROUND(AVG(delivery_delay_days), 2) AS average_delay_days
FROM order_features
GROUP BY delivery_status
ORDER BY order_count DESC;
"""

delivery_df = pd.read_sql(
    delivery_query,
    engine
)

delivery_df

,delivery_status,order_count,average_delivery_days,average_delay_days
0,Early,88626,10.42,-13.71
1,Late,6535,33.49,10.62
2,Not Delivered,2988,NaN,NaN
3,On Time,1292,18.77,0.00


In [20]:
total_delivery_orders = delivery_df["order_count"].sum()

late_orders = delivery_df.loc[
    delivery_df["delivery_status"] == "Late",
    "order_count"
].sum()

late_percentage = (
    late_orders / total_delivery_orders
) * 100

print("DELIVERY PERFORMANCE")

print(
    f"Late orders: {late_orders:,} "
    f"({late_percentage:.2f}% of analyzed orders)"
)

print("\nOBSERVATION")
print(
    f"{late_percentage:.2f}% of the analyzed orders are classified "
    "as late."
)

print("\nINTERPRETATION")
print(
    "Delivery performance varies across orders, indicating that "
    "some orders experience delays relative to their estimated "
    "delivery dates."
)

print("\nBUSINESS IMPACT")
print(
    "Delivery delays can affect customer experience and may increase "
    "the risk of negative customer reviews."
)

print("\nRECOMMENDATION")
print(
    "The business should monitor late deliveries by location, seller, "
    "and operational segment to identify logistics and fulfillment "
    "bottlenecks."
)

DELIVERY PERFORMANCE
Late orders: 6,535 (6.57% of analyzed orders)

OBSERVATION
6.57% of the analyzed orders are classified as late.

INTERPRETATION
Delivery performance varies across orders, indicating that some orders experience delays relative to their estimated delivery dates.

BUSINESS IMPACT
Delivery delays can affect customer experience and may increase the risk of negative customer reviews.

RECOMMENDATION
The business should monitor late deliveries by location, seller, and operational segment to identify logistics and fulfillment bottlenecks.


INSIGHT 9 — DELIVERY VS CUSTOMER SATISFACTION

9. Delivery Performance and Customer Satisfaction

This analysis compares average customer review scores across different
delivery-status groups.

The objective is to understand whether customer satisfaction differs
between early, on-time, and late deliveries.

In [21]:
delivery_review_query = """
SELECT
    o.delivery_status,
    ROUND(AVG(r.review_score), 2) AS average_review_score,
    COUNT(*) AS reviewed_orders
FROM order_features o
JOIN order_reviews r
    ON o.order_id = r.order_id
WHERE r.review_score IS NOT NULL
GROUP BY o.delivery_status
ORDER BY average_review_score DESC;
"""

delivery_review_df = pd.read_sql(
    delivery_review_query,
    engine
)

delivery_review_df

,delivery_status,average_review_score,reviewed_orders
0,Early,4.29,88635
1,On Time,4.03,1291
2,Late,2.27,6410
3,Not Delivered,1.77,2888


In [22]:
print("DELIVERY AND CUSTOMER SATISFACTION")

for _, row in delivery_review_df.iterrows():
    print(
        f"{row['delivery_status']}: "
        f"Average Review Score = {row['average_review_score']:.2f}, "
        f"Reviewed Orders = {row['reviewed_orders']:,}"
    )

highest_rating = delivery_review_df.iloc[0]
lowest_rating = delivery_review_df.iloc[-1]

print("\nOBSERVATION")
print(
    f"The highest average review score among delivery-status groups "
    f"is {highest_rating['average_review_score']:.2f} for "
    f"{highest_rating['delivery_status']} orders, while the lowest "
    f"is {lowest_rating['average_review_score']:.2f} for "
    f"{lowest_rating['delivery_status']} orders."
)

print("\nINTERPRETATION")
print(
    "Differences in review scores across delivery-status groups "
    "indicate that delivery experience may be related to customer "
    "satisfaction."
)

print("\nBUSINESS IMPACT")
print(
    "Poor delivery experiences can reduce customer satisfaction and "
    "may contribute to lower review scores."
)

print("\nRECOMMENDATION")
print(
    "The business should prioritize reducing delivery delays and "
    "investigate locations, sellers, and operational segments where "
    "late deliveries are associated with lower customer ratings."
)

DELIVERY AND CUSTOMER SATISFACTION
Early: Average Review Score = 4.29, Reviewed Orders = 88,635
On Time: Average Review Score = 4.03, Reviewed Orders = 1,291
Late: Average Review Score = 2.27, Reviewed Orders = 6,410
Not Delivered: Average Review Score = 1.77, Reviewed Orders = 2,888

OBSERVATION
The highest average review score among delivery-status groups is 4.29 for Early orders, while the lowest is 1.77 for Not Delivered orders.

INTERPRETATION
Differences in review scores across delivery-status groups indicate that delivery experience may be related to customer satisfaction.

BUSINESS IMPACT
Poor delivery experiences can reduce customer satisfaction and may contribute to lower review scores.

RECOMMENDATION
The business should prioritize reducing delivery delays and investigate locations, sellers, and operational segments where late deliveries are associated with lower customer ratings.


INSIGHT 10 — CUSTOMER SATISFACTION BY CATEGORY

10. Customer Satisfaction by Product Category

Customer review scores are analyzed across product categories to
identify categories associated with relatively higher or lower
customer satisfaction.

Only categories with a sufficient number of reviews are considered
to reduce the influence of categories with very small samples.

In [23]:
category_review_query = """
SELECT
    COALESCE(
        ct.product_category_name_english,
        p.product_category_name,
        'Unknown'
    ) AS category_name,
    ROUND(AVG(r.review_score), 2) AS average_review_score,
    COUNT(*) AS reviewed_orders
FROM order_reviews r
JOIN order_items oi
    ON r.order_id = oi.order_id
JOIN products p
    ON oi.product_id = p.product_id
LEFT JOIN category_translation ct
    ON p.product_category_name = ct.product_category_name
WHERE r.review_score IS NOT NULL
GROUP BY
    COALESCE(
        ct.product_category_name_english,
        p.product_category_name,
        'Unknown'
    )
HAVING COUNT(*) >= 20
ORDER BY average_review_score DESC;
"""

category_review_df = pd.read_sql(
    category_review_query,
    engine
)

category_review_df.head(15)

,category_name,average_review_score,reviewed_orders
0,books_general_interest,4.45,549
1,costruction_tools_tools,4.44,99
2,flowers,4.42,31
3,books_imported,4.40,60
4,books_technical,4.37,266
5,food_drink,4.32,279
6,luggage_accessories,4.32,1088
7,small_appliances_home_oven_and_coffee,4.30,76
8,fashion_sport,4.26,31
9,fashion_shoes,4.23,261


In [24]:
highest_category_rating = category_review_df.iloc[0]
lowest_category_rating = category_review_df.iloc[-1]

print("CUSTOMER SATISFACTION BY CATEGORY")

print(
    f"Highest-rated category: "
    f"{highest_category_rating['category_name']} "
    f"({highest_category_rating['average_review_score']:.2f}/5)"
)

print(
    f"Lowest-rated category: "
    f"{lowest_category_rating['category_name']} "
    f"({lowest_category_rating['average_review_score']:.2f}/5)"
)

print("\nOBSERVATION")
print(
    f"Average review scores vary across product categories, with "
    f"{highest_category_rating['category_name']} receiving the "
    f"highest average score and {lowest_category_rating['category_name']} "
    f"receiving the lowest among categories meeting the review-count "
    f"threshold."
)

print("\nINTERPRETATION")
print(
    "Customer satisfaction is not uniform across product categories."
)

print("\nBUSINESS IMPACT")
print(
    "Lower-rated categories may contain product quality, seller, "
    "delivery, or customer-expectation issues that require further "
    "investigation."
)

print("\nRECOMMENDATION")
print(
    "The business should investigate low-rated categories using "
    "product-level performance, seller ratings, delivery performance, "
    "and customer review comments."
)

CUSTOMER SATISFACTION BY CATEGORY
Highest-rated category: books_general_interest (4.45/5)
Lowest-rated category: diapers_and_hygiene (3.26/5)

OBSERVATION
Average review scores vary across product categories, with books_general_interest receiving the highest average score and diapers_and_hygiene receiving the lowest among categories meeting the review-count threshold.

INTERPRETATION
Customer satisfaction is not uniform across product categories.

BUSINESS IMPACT
Lower-rated categories may contain product quality, seller, delivery, or customer-expectation issues that require further investigation.

RECOMMENDATION
The business should investigate low-rated categories using product-level performance, seller ratings, delivery performance, and customer review comments.


INSIGHT 11 — T-TEST

11. Statistical Finding - Independent Two-Sample T-Test

Business Question

Do delayed orders receive significantly different review scores
compared with on-time orders?

 Null Hypothesis (H0)

There is no significant difference in average review scores between
delayed and on-time orders.

 Alternative Hypothesis (H1)

There is a significant difference in average review scores between
delayed and on-time orders.

The significance level is set at 5%.

In [25]:
from scipy.stats import ttest_ind

ttest_query = """
SELECT
    o.delivery_status,
    r.review_score
FROM order_features o
JOIN order_reviews r
    ON o.order_id = r.order_id
WHERE r.review_score IS NOT NULL
  AND o.delivery_status IN ('Late', 'On Time');
"""

ttest_df = pd.read_sql(
    ttest_query,
    engine
)

delayed_scores = ttest_df.loc[
    ttest_df["delivery_status"] == "Late",
    "review_score"
].dropna()

on_time_scores = ttest_df.loc[
    ttest_df["delivery_status"] == "On Time",
    "review_score"
].dropna()

t_stat, p_value = ttest_ind(
    delayed_scores,
    on_time_scores,
    equal_var=False
)

print("Independent Two-Sample T-Test")
print("--------------------------------")
print(f"Delayed orders mean review score: {delayed_scores.mean():.3f}")
print(f"On-time orders mean review score: {on_time_scores.mean():.3f}")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.6f}")

alpha = 0.05

if p_value < alpha:
    decision = "Reject the null hypothesis."
    conclusion = (
        "There is statistically significant evidence of a difference "
        "in review scores between delayed and on-time orders."
    )
else:
    decision = "Fail to reject the null hypothesis."
    conclusion = (
        "There is not sufficient statistical evidence of a difference "
        "in review scores between delayed and on-time orders."
    )

print(f"\nDecision: {decision}")
print(f"Business conclusion: {conclusion}")

Independent Two-Sample T-Test
--------------------------------
Delayed orders mean review score: 2.271
On-time orders mean review score: 4.033
T-statistic: -44.1021
P-value: 0.000000

Decision: Reject the null hypothesis.
Business conclusion: There is statistically significant evidence of a difference in review scores between delayed and on-time orders.


INSIGHT 12 — ANOVA

12. Statistical Finding - One-Way ANOVA

Business Question

Does average order value differ significantly across product categories?

 Null Hypothesis (H0)

There is no significant difference in average order value among
product categories.

Alternative Hypothesis (H1)

At least one product category has a significantly different average
order value.

The significance level is set at 5%.

In [26]:
from scipy.stats import f_oneway

anova_query = """
SELECT
    COALESCE(
        ct.product_category_name_english,
        p.product_category_name,
        'Unknown'
    ) AS category_name,
    oi.order_id,
    SUM(oi.price + oi.freight_value) AS order_value
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
LEFT JOIN category_translation ct
    ON p.product_category_name = ct.product_category_name
GROUP BY
    category_name,
    oi.order_id;
"""

anova_df = pd.read_sql(
    anova_query,
    engine
)

category_groups = []

for category, group in anova_df.groupby("category_name"):
    if len(group) >= 20:
        category_groups.append(
            group["order_value"].dropna()
        )

f_stat, p_value = f_oneway(*category_groups)

print("One-Way ANOVA")
print("-------------")
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.6f}")

alpha = 0.05

if p_value < alpha:
    print("\nDecision: Reject the null hypothesis.")
    print(
        "Business conclusion: There is statistically significant "
        "evidence that average order value differs across at least "
        "some product categories."
    )
else:
    print("\nDecision: Fail to reject the null hypothesis.")
    print(
        "Business conclusion: There is not sufficient statistical "
        "evidence that average order value differs across product "
        "categories."
    )

One-Way ANOVA
-------------
F-statistic: 162.9888
P-value: 0.000000

Decision: Reject the null hypothesis.
Business conclusion: There is statistically significant evidence that average order value differs across at least some product categories.


INSIGHT 13 — CHI-SQUARE

13. Statistical Finding - Chi-Square Test

Business Question

Is there a significant association between payment method and order
status?

Null Hypothesis (H0)

Payment method and order status are independent.

 Alternative Hypothesis (H1)

Payment method and order status are associated.

The significance level is set at 5%.

In [27]:
from scipy.stats import chi2_contingency

chi_query = """
SELECT
    op.payment_type,
    o.order_status
FROM order_payments op
JOIN orders o
    ON op.order_id = o.order_id;
"""

chi_df = pd.read_sql(
    chi_query,
    engine
)

contingency_table = pd.crosstab(
    chi_df["payment_type"],
    chi_df["order_status"]
)

chi2_stat, p_value, dof, expected = chi2_contingency(
    contingency_table
)

print("Chi-Square Test of Independence")
print("--------------------------------")
print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"P-value: {p_value:.6f}")

alpha = 0.05

if p_value < alpha:
    print("\nDecision: Reject the null hypothesis.")
    print(
        "Business conclusion: There is statistically significant "
        "evidence of an association between payment method and "
        "order status."
    )
else:
    print("\nDecision: Fail to reject the null hypothesis.")
    print(
        "Business conclusion: There is not sufficient statistical "
        "evidence of an association between payment method and "
        "order status."
    )

Chi-Square Test of Independence
--------------------------------
Chi-square statistic: 677.0831
Degrees of freedom: 28
P-value: 0.000000

Decision: Reject the null hypothesis.
Business conclusion: There is statistically significant evidence of an association between payment method and order status.


 14. Overall Business Recommendations

Based on the combined findings from sales analysis, customer analysis,
seller performance, delivery analysis, customer experience, SQL
analysis, and statistical testing, the following recommendations can
be considered.

 1. Strengthen Customer Retention

The customer analysis identifies repeat and one-time customers as
distinct customer groups.

The business should focus on increasing repeat purchases through
personalized recommendations, targeted offers, loyalty initiatives,
and post-purchase engagement.

---

 2. Prioritize High-Value Customers

Customers with higher total spending represent an important source of
business value.

The business should identify high-value customers and develop targeted
retention strategies to encourage continued purchases.

---

 3. Focus on High-Performing Product Categories

Revenue is not distributed equally across all product categories.

High-performing categories should receive appropriate inventory,
marketing, and seller support.

Lower-performing categories should be investigated to determine
whether demand, pricing, product availability, or customer experience
is limiting their performance.

---

 4. Protect High-Revenue Products

High-performing products are important contributors to revenue.

The business should monitor their availability, seller performance,
and delivery performance to reduce the risk of lost sales.

---

 5. Improve Delivery Performance

Delivery analysis identifies orders that are early, on time, late,
or not delivered.

The business should monitor delivery delays by seller and location
and investigate operational bottlenecks affecting delayed orders.

---

 6. Improve Customer Experience

Customer reviews provide an important measure of customer satisfaction.

Categories and operational segments with lower review scores should
be investigated using product, seller, delivery, and review information.

---

 7. Monitor Seller Performance

Seller revenue, order volume, product sales, and customer ratings should
be monitored together.

High-performing sellers can be supported and encouraged to expand,
while sellers with weaker performance can be evaluated for operational
or product-related issues.

---

 8. Use Data-Driven Planning

Monthly revenue trends can be used to support inventory planning,
marketing campaigns, seller capacity planning, and logistics preparation.

The business should regularly monitor these trends through the
interactive dashboard.

 15. Summary

The Cart2Insights analysis provides a comprehensive view of e-commerce
business performance across sales, customers, products, sellers,
delivery, and customer experience.

Key Findings

1. **Overall Business Performance**

   Revenue, order volume, customer count, seller count, average order
   value, and review scores provide the primary indicators of overall
   business performance.

2. **Sales Performance**

   Monthly revenue analysis identifies periods of stronger and weaker
   sales activity, providing useful information for business planning.

3. **Product Categories**

   Revenue and sales are distributed unevenly across product categories,
   with certain categories contributing more strongly to overall
   business performance.

4. **Top Products**

   A group of products generates higher revenue than other products,
   making their availability and seller performance important.

5. **Customer Value**

   Customer spending varies across the customer base, with high-value
   customers representing an important segment for retention efforts.

6. **Customer Retention**

   The repeat-customer analysis provides an indication of customer
   loyalty and highlights opportunities for improving repeat purchases.

7. **Seller Performance**

   Seller performance varies by revenue, order volume, and items sold,
   making seller-level monitoring important for marketplace performance.

8. **Delivery Performance**

   Delivery status analysis identifies differences between early,
   on-time, late, and undelivered orders.

9. **Customer Satisfaction**

   Review scores vary across delivery-status groups and product
   categories, highlighting the importance of customer experience
   monitoring.

10. **Statistical Evidence**

    Hypothesis testing using the independent two-sample t-test,
    one-way ANOVA, and chi-square test provides statistical support
    for evaluating relationships and differences identified during
    the analysis.

### Overall Recommendation

The business should combine sales, customer, seller, product, delivery,
and customer-experience metrics when making decisions. The Streamlit
dashboard can be used as an interactive monitoring tool to identify
changes in business performance and investigate areas requiring
attention.

16. Business Insights Summary

| Business Area | Observation | Business Impact | Recommendation |
|---|---|---|---|
| Sales | Revenue varies across time and business segments. | Revenue performance requires continuous monitoring. | Use revenue trends for planning and resource allocation. |
| Products | Product performance varies significantly. | High-performing products have greater revenue importance. | Protect availability of high-performing products. |
| Categories | Category revenue is unevenly distributed. | Certain categories have greater influence on revenue. | Prioritize high-performing categories and investigate weaker ones. |
| Customers | Customer spending varies across the customer base. | High-value customers represent an important revenue segment. | Develop targeted retention strategies. |
| Repeat Customers | Customers differ in repeat purchasing behavior. | Repeat purchases support customer retention and long-term value. | Encourage repeat purchases through personalized engagement. |
| Sellers | Seller performance varies in revenue and order volume. | Seller performance affects marketplace revenue. | Monitor and support sellers based on multiple KPIs. |
| Delivery | Some orders experience delivery delays. | Delays may affect customer experience. | Investigate delay patterns by location and seller. |
| Customer Experience | Review scores vary across delivery and product categories. | Customer satisfaction is not uniform. | Investigate low-rated categories and operational segments. |
| Statistics | Hypothesis tests provide evidence about key relationships. | Statistical evidence improves confidence in business conclusions. | Use statistical findings alongside descriptive analysis. |

 17. Conclusion

Cart2Insights demonstrates an end-to-end e-commerce analytics workflow
using Python, Pandas, MySQL, SQL, statistical analysis, feature
engineering, and Streamlit.

The project transforms raw multi-table e-commerce data into structured
business information and actionable insights.

The analysis covers:

- Sales performance
- Revenue trends
- Product and category performance
- Customer behavior
- Customer retention
- Seller performance
- Delivery efficiency
- Customer satisfaction
- Statistical relationships

The final recommendations focus on customer retention, high-value
customers, product and category performance, seller management,
delivery optimization, and customer experience improvement.

The Streamlit dashboard provides an interactive interface through which
business users can explore these findings using filters and
visualizations.